# Práctica de Procesamiento de Lenguaje Natural (PLN)
## Modelos de Lenguaje con Redes Recurrentes — Elman RNN

---

### ¿Qué vamos a hacer en esta práctica?

Construiremos **desde cero** un modelo de lenguaje basado en una **Red Neuronal Recurrente de Elman (1990)** capaz de generar texto en español a partir de letras de canciones.

| Paso | Descripción |
|------|-------------|
| 1 | **Lectura y tokenización** del corpus |
| 2 | **Construcción del vocabulario** (palabra → tokenID) |
| 3 | **Preparación del dataset** para modelado de lenguaje (inputs / targets desplazados) |
| 4 | **Definición del modelo** Elman RNN |
| 5 | **Dataset y DataLoader** de PyTorch con padding dinámico |
| 6 | **Entrenamiento** del modelo |
| 7 | **Generación de texto** con muestreo Top-K autoregresivo |

---

### Comparativa con los modelos anteriores

| | Bengio (2003) | **Elman RNN** |
|---|---|---|
| Tarea | Predicción siguiente palabra | **Predicción siguiente palabra** |
| Ventana de contexto | Fija (`context_size`) | **Variable — toda la historia** |
| Memoria | Ninguna | **Estado oculto recurrente** |
| Secuencia entrada | Contexto fijo | **Toda la secuencia** |
| Complejidad temporal | O(1) ventana fija | **O(T)** — depende de la longitud T de la frase |

### El modelo: Red de Elman (1990)

Jeffrey Elman propuso en 1990 una arquitectura en la que la red procesa el texto **token a token**, manteniendo un **estado oculto** que actúa como memoria de lo que ha visto hasta ese momento.

```
t=1           t=2           t=3
 x₁ ──► RNN ──► x₂ ──► RNN ──► x₃ ──► RNN ──► ...
          │               │               │
          h₁ ─────────────h₂ ─────────────h₃
          │               │               │
         ŷ₁              ŷ₂              ŷ₃
```

La idea central: **el estado oculto hₜ codifica toda la historia de tokens anteriores**, permitiendo predicciones que dependen de contextos arbitrariamente largos.

---


## PASO 0: Instalación e Importaciones

Comprobamos que PyTorch está disponible e importamos todas las librerías necesarias.


In [4]:
# En Google Colab PyTorch ya está disponible.
# Si trabajas en local y no lo tienes, descomenta la siguiente línea:
# !pip install torch

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from functools import partial
import re

print(f'PyTorch versión: {torch.__version__}')
print(f'GPU disponible:  {torch.cuda.is_available()}')
print('OK - Librerías importadas correctamente.')


PyTorch versión: 2.11.0+cu130
GPU disponible:  True
OK - Librerías importadas correctamente.


---
## PASO 1: Lectura y Tokenización del Corpus

### Corpus para entrenamiento
Un **corpus** es una colección de textos que usamos para entrenar el modelo. En esta práctica son letras de canciones.

### Formato del archivo
Cada canción está delimitada por las etiquetas `<CS>` y `</CS>`. Dentro de cada canción, cada verso termina con el token especial `<EOL>` (End Of Line):

```
<CS>
Ya te dije frases hechas y deshechas <EOL>
Ya he soñado algún momento de tu piel <EOL>
</CS>
```

> **Reutilizamos directamente la función `read_and_tokenize` de la práctica anterior** (Bengio generativo).  
> El formato del corpus es idéntico, por lo que no necesitamos ningún cambio.

### Resultado esperado
```python
[
  ['Ya', 'te', 'dije', 'frases', ..., '<EOL>', ...],
  ['Vida', 'de', 'mi', 'vida', '<EOL>', ...],
  ...
]
```


In [5]:
import re

def read_and_tokenize(filename):
    """
    Lee un corpus de canciones con formato <CS>...</CS> y lo tokeniza.

    Tokens especiales preservados: <CS>, </CS>, <EOL>
    El resto de puntuación se separa en tokens individuales.
    """
    with open(filename, 'r', encoding='utf-8-sig') as f:
        content = f.read()

    # Patrón que captura: <CS>, </CS>, <EOL>, palabras, y puntuación individual
    tokens = re.findall(r'<CS>|</CS>|<EOL>|\w+|[^\w\s<>/]', content)

    # Agrupamos por canciones: iniciamos nueva lista cada vez que vemos <CS>
    sentences = []
    current_song = []

    for token in tokens:
        if token == '<CS>':
            # Si hay canción en curso, la guardamos (por si hay CS anidados o sueltos)
            if current_song:
                sentences.append(current_song)
            current_song = []
        elif token == '</CS>':
            if current_song:
                sentences.append(current_song)
            current_song = []
        else:
            current_song.append(token)

    # Por si queda algo sin cerrar
    if current_song:
        sentences.append(current_song)

    return sentences

# Carga del corpus
Actualiza la variable path_corpus con tu corpus de trabajo




In [6]:
# ── Configura aquí la fuente de datos ────────────────────────────────────────

path_corpus = 'corpusA.txt'
sentences = read_and_tokenize(path_corpus)

print(f'Lectura y tokenización del corpus finalizada.')
print(f"🔢 Total de sentencias en el corpus: {len(sentences)}")


sentences = sentences[:1000]  # Primeras 1000 canciones para agilizar


# -----------------------------------------------------------------------
print(f'Total de canciones cargadas: {len(sentences)}')
print(f'\nPrimera cancion ({len(sentences[0])} tokens):')
print(sentences[0])
print(f'\nTotal de tokens en el corpus: {sum(len(s) for s in sentences)}')


Lectura y tokenización del corpus finalizada.
🔢 Total de sentencias en el corpus: 6606
Total de canciones cargadas: 1000

Primera cancion (184 tokens):
['Tú', 'y', 'yo', 'en', 'mi', 'habitación', '<EOL>', 'Y', 'el', 'mundo', 'fuera', '<EOL>', 'Puede', 'que', 'parezca', 'una', 'noche', 'más', '<EOL>', 'Pero', 'yo', 'sé', 'que', 'así', 'esparzo', 'amor', '<EOL>', 'Y', 'mañana', 'cuando', 'salga', 'el', 'sol', '<EOL>', 'Nos', 'querremos', 'aún', 'más', '<EOL>', 'Y', 'al', 'fin', 'te', 'podré', 'abrazar', '<EOL>', 'Como', 'lo', 'hacíamos', 'antes', '<EOL>', 'Yo', 'me', 'siento', 'solo', 'por', 'sólo', 'un', 'instante', '<EOL>', 'Pero', 'sé', 'que', 'cuando', 'pase', 'todo', 'esto', '<EOL>', 'Tú', 'y', 'yo', 'volveremos', 'a', 'ser', 'como', 'éramos', 'antes', '<EOL>', 'Y', 'cuando', 'ya', 'esté', 'seguro', 'que', 'no', 'pueda', 'herir', 'a', 'nadie', '<EOL>', 'Entonces', 'mis', 'pasos', 'me', 'llevarán', 'fuera', 'de', 'este', 'caso', '<EOL>', 'Me', 'acercarán', 'hasta', 'donde', 'tú', 'es

---
## PASO 2: Construcción del Vocabulario

### Reutilizamos la clase `Vocabulary`

La clase es **idéntica** a la del modelo Bengio generativo.  
La única diferencia es que aquí añadimos `<PAD>` como **primer token** (índice 0).

> **¿Por qué `<PAD>` en un modelo generativo?**  
> La RNN procesa secuencias de longitud variable. Cuando agrupamos varias canciones en un batch,  
> las más cortas se rellenan hasta la longitud de la más larga. El modelo debe ignorar esos tokens  
> de relleno al calcular la pérdida (lo veremos en el Paso 6).

### Comparativa de tokens especiales

| Token | Bengio generativo | Elman RNN |
|---|---|---|
| `<PAD>` | Sí (relleno de contexto) | Sí (relleno de batch) |
| `<EOL>` | Sí (ya en el corpus) | Sí (ya en el corpus) |


In [7]:
class Vocabulary:
    """
    Diccionario bidireccional: palabra <-> índice numérico.

    Reutilizada de la práctica Bengio — sin cambios.

    Atributos:
        token_to_idx (dict): 'vida' -> 5
        idx_to_token (dict):  5 -> 'vida'
        size (int): número total de tokens registrados
    """

    def __init__(self):
        self.token_to_idx = {}
        self.idx_to_token = {}
        self.size = 0

    def add_token(self, token):
        """Añade un token al vocabulario si no existe ya."""
        if token not in self.token_to_idx:
            self.token_to_idx[token] = self.size
            self.idx_to_token[self.size] = token
            self.size += 1

    def encode(self, tokens):
        """
        Convierte lista de palabras en lista de índices numéricos.
        Las palabras desconocidas se mapean al índice de <PAD> (0).
        """
        return [self.token_to_idx.get(tok, 0) for tok in tokens]

    def decode(self, indices):
        """Convierte lista de índices en lista de palabras."""
        return [self.idx_to_token[idx] for idx in indices]


# Construimos el vocabulario
vocab = Vocabulary()
vocab.add_token('<PAD>')   # índice 0 — siempre primero

for sentence in sentences:
    for token in sentence:
        vocab.add_token(token)

print(f'Vocabulario construido.')
print(f'  Tamaño total : {vocab.size} tokens únicos')
print(f'  Índice <PAD> : {vocab.token_to_idx["<PAD>"]}')
print(f'  Índice <EOL> : {vocab.token_to_idx.get("<EOL>", "No encontrado")}')

# Ejemplo de codificación/decodificación
ejemplo = sentences[0][:5]
print(f'\nEjemplo encode: {ejemplo}')
print(f'                -> {vocab.encode(ejemplo)}')
print(f'Ejemplo decode: {vocab.encode(ejemplo)} -> {vocab.decode(vocab.encode(ejemplo))}')


Vocabulario construido.
  Tamaño total : 14962 tokens únicos
  Índice <PAD> : 0
  Índice <EOL> : 7

Ejemplo encode: ['Tú', 'y', 'yo', 'en', 'mi']
                -> [1, 2, 3, 4, 5]
Ejemplo decode: [1, 2, 3, 4, 5] -> ['Tú', 'y', 'yo', 'en', 'mi']


---
## PASO 3: Preparación del Dataset (Secuencias Desplazadas)

### ¿Cómo aprende la RNN a predecir la siguiente palabra?

La estrategia es diferente a la de Bengio. En lugar de ventanas de contexto fijas, la RNN procesa **la canción entera** de izquierda a derecha.

Para una canción `[w₀, w₁, w₂, w₃, w₄]` creamos **un único par**:

```
ENTRADA (inputs):   [w₀, w₁, w₂, w₃]   <- todos menos el último
OBJETIVO (targets): [w₁, w₂, w₃, w₄]   <- todos menos el primero  (desplazado 1 posición)
```

En cada paso de tiempo `t`, la RNN recibe `wₜ` y debe predecir `wₜ₊₁`.

### Comparativa de estrategias de dataset

| | Bengio generativo | **Elman RNN** |
|---|---|---|
| Ejemplos por canción | Uno por cada token (ventana deslizante) | **Uno por canción completa** |
| Longitud de entrada | Fija (`context_size`) | **Variable (longitud de canción - 1)** |
| Contexto disponible | Solo N palabras anteriores | **Toda la historia hasta tₙ** |
| Nº total de ejemplos | `Σ len(canción)` | **`len(canciones)`** |

> **¿Por qué menos ejemplos pero igual de eficiente?**  
> Aunque hay menos pares (canción, secuencia), en cada paso de entrenamiento  
> la RNN produce una predicción por cada token de la secuencia.  
> Una canción de 50 tokens genera 49 predicciones en un solo forward pass.


In [8]:
def prepare_rnn_dataset(sentences, vocab):
    """
    Prepara el dataset para el modelo ElmanRNN.

    Para cada canción crea exactamente UN par:
        inputs  = canción[:-1]  (todos los tokens excepto el último)
        targets = canción[1:]   (todos los tokens excepto el primero)

    La RNN en el paso t recibe inputs[t] y debe predecir targets[t] = inputs[t+1].

    Args:
        sentences (list):   Lista de canciones tokenizadas.
        vocab (Vocabulary): Vocabulario ya construido.

    Returns:
        list[tuple]: Lista de pares (lista_inputs_ids, lista_targets_ids).
    """
    data = []
    for sentence in sentences:
        encoded = vocab.encode(sentence)   # Codificamos toda la canción
        if len(encoded) < 2:
            continue   # Ignoramos canciones de menos de 2 tokens
        inputs  = encoded[:-1]   # Todos menos el último
        targets = encoded[1:]    # Todos menos el primero (desplazado 1)
        data.append((inputs, targets))
    return data


rnn_data = prepare_rnn_dataset(sentences, vocab)

print(f'Dataset preparado.')
print(f'  Canciones procesadas : {len(rnn_data)}')
print(f'  (Una por canción — no ventanas deslizantes)')

print(f'\nPrimera canción:')
inputs_ex, targets_ex = rnn_data[0]
print(f'  Longitud inputs  : {len(inputs_ex)}')
print(f'  Longitud targets : {len(targets_ex)}')

print(f'\nPrimeros 5 pares (input -> target):')
print(f'  {"INPUT":20s}  TARGET')
print('  ' + '-' * 38)
for i in range(min(5, len(inputs_ex))):
    inp_word = vocab.idx_to_token[inputs_ex[i]]
    tgt_word = vocab.idx_to_token[targets_ex[i]]
    print(f'  {inp_word:20s}  {tgt_word}')


Dataset preparado.
  Canciones procesadas : 1000
  (Una por canción — no ventanas deslizantes)

Primera canción:
  Longitud inputs  : 183
  Longitud targets : 183

Primeros 5 pares (input -> target):
  INPUT                 TARGET
  --------------------------------------
  Tú                    y
  y                     yo
  yo                    en
  en                    mi
  mi                    habitación


---
## PASO 4: Definición del Modelo Elman RNN

### Arquitectura completa

```
ENTRADA en t:  xₜ  (índice de token)
                │
    +-----------+----------+
    |   EMBEDDING          |  xₜ (índice) -> eₜ (vector embed_size)
    |   nn.Embedding       |  Igual que en Bengio, pero ahora procesamos
    +-----------+----------+  UN token a la vez, no un contexto fijo
                │
    +-----------+----------+
    |   CAPA RNN           |  Recibe: eₜ y el estado oculto anterior hₜ₋₁
    |   nn.RNN             |  Calcula: hₜ = tanh(Wₓ·eₜ + Wₕ·hₜ₋₁ + b)
    +-----------+----------+  El estado hₜ codifica toda la historia hasta t
                │
    +-----------+----------+
    |  CAPA DE SALIDA      |  hₜ -> logits de tamaño vocab_size
    |  nn.Linear           |  Una puntuación por cada palabra del vocabulario
    +-----------+----------+
                │
SALIDA en t:  [0.1, -2.3, 5.7, ...]  <- vocab_size logits
              La palabra con mayor logit es la predicción para tₜ₊₁
```

### La ecuación de la RNN de Elman

```
hₜ = tanh(Wₓ · eₜ  +  Wₕ · hₜ₋₁  +  b)
          ─────────    ──────────────
          contribución  contribución de
          del token     la historia pasada
          actual
```

- `Wₓ` — matriz de pesos para la entrada (`embed_size × hidden_dim`)
- `Wₕ` — matriz de pesos para el estado oculto recurrente (`hidden_dim × hidden_dim`)
- `tanh` — activación que mantiene los valores entre -1 y +1

### Diferencias clave respecto a Bengio

| Aspecto | Bengio | **Elman RNN** |
|---|---|---|
| Parámetro de contexto | `context_size × embed_size → hidden` | **`embed_size + hidden_dim → hidden`** |
| Memoria | Ninguna (ventana fija) | **Estado oculto recurrente hₜ** |
| Módulo PyTorch | `nn.Linear` + concatenación | **`nn.RNN`** |
| Entrada a fc_out | `hidden_dim` | **`hidden_dim` (igual)** |
| Salida | `vocab_size` logits | **`vocab_size` logits por cada t** |


In [9]:
class ElmanRNN(nn.Module):
    """
    Red Neuronal Recurrente de Elman (1990) para modelado de lenguaje.

    Dado el token actual, predice el siguiente manteniendo un estado oculto
    que actúa como memoria de toda la historia previa.

    Arquitectura:
        Embedding -> nn.RNN -> Capa de salida (logits para cada vocab)

    Args:
        vocab_size (int):  Número de palabras en el vocabulario.
        embed_size (int):  Dimensión de los vectores de embedding.
        hidden_dim (int):  Dimensión del estado oculto de la RNN.
    """

    def __init__(self, vocab_size, embed_size, hidden_dim):
        super(ElmanRNN, self).__init__()

        # Tabla de embeddings: igual que en Bengio
        # vocab_size filas, cada una de embed_size dimensiones
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Capa RNN de Elman
        # - input_size  = embed_size  (tamaño del vector de embedding)
        # - hidden_size = hidden_dim  (tamaño del estado oculto)
        # - batch_first = True        (esperamos [batch, seq_len, features])
        self.rnn = nn.RNN(embed_size, hidden_dim, batch_first=True)

        # Capa de salida: un logit por cada palabra del vocabulario
        # Se aplica a CADA posición temporal de la secuencia
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        """
        Propagación hacia adelante.

        Args:
            x      : Tensor [batch, seq_len] — índices de tokens de entrada.
            hidden : Tensor [1, batch, hidden_dim] — estado oculto inicial.
                     Si es None, PyTorch inicializa con ceros.

        Returns:
            logits : Tensor [batch, seq_len, vocab_size] — logits por posición.
            hidden : Tensor [1, batch, hidden_dim] — estado oculto final.
        """
        # 1. Lookup de embeddings
        #    [batch, seq_len] -> [batch, seq_len, embed_size]
        emb = self.embedding(x)

        # 2. Capa RNN — procesa toda la secuencia de una vez (internamente t a t)
        #    output : [batch, seq_len, hidden_dim]  — estados ocultos en cada t
        #    hidden : [1, batch, hidden_dim]         — estado oculto en el último t
        output, hidden = self.rnn(emb, hidden)

        # 3. Capa de salida: aplicamos fc a CADA estado oculto
        #    [batch, seq_len, hidden_dim] -> [batch, seq_len, vocab_size]
        logits = self.fc(output)

        return logits, hidden


# ── Hiperparámetros del modelo ────────────────────────────────────────────────
EMBED_SIZE = 128
HIDDEN_DIM = 256

model = ElmanRNN(
    vocab_size = vocab.size,
    embed_size = EMBED_SIZE,
    hidden_dim = HIDDEN_DIM
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
emb_params   = vocab.size * EMBED_SIZE
rnn_params   = total_params - emb_params - (HIDDEN_DIM * vocab.size + vocab.size)

print('Modelo ElmanRNN creado.')
print(f'\nArquitectura:')
print(model)
print(f'\nParámetros entrenables: {total_params:,}')
print(f'  - Embedding table : {vocab.size} × {EMBED_SIZE}   = {vocab.size * EMBED_SIZE:,}')
print(f'  - RNN (Wx + Wh)   : ~{rnn_params:,}')
print(f'  - fc (salida)     : {HIDDEN_DIM} × {vocab.size} = {HIDDEN_DIM * vocab.size:,}')

# Prueba rápida de las dimensiones del forward pass
dummy_input = torch.zeros(2, 10, dtype=torch.long)   # batch=2, seq_len=10
logits_test, hidden_test = model(dummy_input)
print(f'\nTest de dimensiones (batch=2, seq_len=10):')
print(f'  logits : {logits_test.shape}   (batch × seq_len × vocab_size)')
print(f'  hidden : {hidden_test.shape}     (1 × batch × hidden_dim)')


Modelo ElmanRNN creado.

Arquitectura:
ElmanRNN(
  (embedding): Embedding(14962, 128)
  (rnn): RNN(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=14962, bias=True)
)

Parámetros entrenables: 5,859,186
  - Embedding table : 14962 × 128   = 1,915,136
  - RNN (Wx + Wh)   : ~98,816
  - fc (salida)     : 256 × 14962 = 3,830,272

Test de dimensiones (batch=2, seq_len=10):
  logits : torch.Size([2, 10, 14962])   (batch × seq_len × vocab_size)
  hidden : torch.Size([1, 2, 256])     (1 × batch × hidden_dim)


---
## PASO 5: Dataset y DataLoader con Padding Dinámico

### El problema: secuencias de longitud variable

A diferencia del modelo Bengio, donde cada ejemplo tiene exactamente `context_size` tokens,  
aquí cada canción tiene su propia longitud. PyTorch no puede apilar tensores de distintas longitudes en un batch.

**Solución: padding dinámico por batch**  
Se rellena cada secuencia del batch hasta la longitud de la **más larga del batch**.  
Esto lo hace automáticamente `pad_sequence`.

```
Canción A: [tok0, tok1, tok2, tok3, tok4]           longitud 5
Canción B: [tok0, tok1, tok2]                        longitud 3
Canción C: [tok0, tok1, tok2, tok3, tok4, tok5, tok6] longitud 7

Batch resultante (pad hasta 7):
  A: [tok0, tok1, tok2, tok3, tok4, PAD,  PAD ]
  B: [tok0, tok1, tok2, PAD,  PAD,  PAD,  PAD ]
  C: [tok0, tok1, tok2, tok3, tok4, tok5, tok6]
```

### `collate_fn`: cómo se ensambla cada batch

`collate_fn` es una función que `DataLoader` llama para convertir una lista de ejemplos  
en un batch. La versión estándar no sabe manejar tensores de distinto tamaño; la nuestra sí.

> **Reutilizamos `CustomDataset` y `rnn_collate_fn` de `ElmanRNN.py`** sin cambios.  
> Usamos `functools.partial` para inyectar el vocabulario en la función de collate.


In [10]:
class CustomDataset(Dataset):
    """
    Envuelve la lista de pares (inputs, targets) en el formato que PyTorch necesita.

    Reutilizada de ElmanRNN.py — sin cambios.
    """

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        inputs, targets = self.data[idx]
        # dtype=torch.long porque son índices enteros (requerido por nn.Embedding)
        return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)


def rnn_collate_fn(batch, vocab):
    """
    Función de collate con padding dinámico para secuencias de longitud variable.

    Reutilizada de ElmanRNN.py.

    Ensambla una lista de pares (inputs, targets) en dos tensores con padding:
        inputs  : [batch_size, max_len]  relleno con pad_idx al final
        targets : [batch_size, max_len]  relleno con pad_idx al final

    Args:
        batch (list): Lista de tuplas (tensor_inputs, tensor_targets).
        vocab (Vocabulary): Vocabulario para obtener el índice de <PAD>.
    """
    inputs, targets = zip(*batch)
    pad_idx = vocab.token_to_idx['<PAD>']

    # pad_sequence agrupa los tensores y rellena los más cortos al final
    inputs  = torch.nn.utils.rnn.pad_sequence(inputs,  batch_first=True, padding_value=pad_idx)
    targets = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=pad_idx)

    return inputs, targets


# ── DataLoader ────────────────────────────────────────────────────────────────
BATCH_SIZE = 4   # Pequeño porque el corpus de ejemplo es pequeño

# partial inyecta el vocabulario en rnn_collate_fn sin tener que modificar su firma
collate_fn_with_vocab = partial(rnn_collate_fn, vocab=vocab)

rnn_dataset = CustomDataset(rnn_data)
rnn_loader  = DataLoader(
    rnn_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    collate_fn  = collate_fn_with_vocab
)

print(f'Dataset y DataLoader listos.')
print(f'  Ejemplos totales  : {len(rnn_dataset)}')
print(f'  Batch size        : {BATCH_SIZE}')
print(f'  Batches por época : {len(rnn_loader)}')

# Inspeccionamos un batch real
sample_inputs, sample_targets = next(iter(rnn_loader))
print(f'\nForma tensor inputs  : {sample_inputs.shape}  (batch × max_len_del_batch)')
print(f'Forma tensor targets : {sample_targets.shape}')
print(f'\nPrimer ejemplo del batch (inputs):')
print(f'  IDs     : {sample_inputs[0].tolist()}')
print(f'  Palabras: {vocab.decode(sample_inputs[0].tolist())}')
print(f'\nPrimer ejemplo del batch (targets — desplazado 1):')
print(f'  IDs     : {sample_targets[0].tolist()}')
print(f'  Palabras: {vocab.decode(sample_targets[0].tolist())}')


Dataset y DataLoader listos.
  Ejemplos totales  : 1000
  Batch size        : 4
  Batches por época : 250

Forma tensor inputs  : torch.Size([4, 359])  (batch × max_len_del_batch)
Forma tensor targets : torch.Size([4, 359])

Primer ejemplo del batch (inputs):
  IDs     : [5537, 9306, 2, 13795, 8025, 96, 2, 48, 9, 1123, 7, 8658, 2, 17, 1452, 102, 13796, 7, 508, 3254, 66, 9576, 4, 9, 308, 7, 8, 255, 482, 13797, 4, 9, 802, 7, 644, 600, 104, 13798, 96, 6684, 2, 11328, 7, 508, 470, 96, 255, 8502, 2, 9, 13799, 7, 8, 45, 4293, 66, 13800, 2, 66, 13801, 7, 81, 40, 1820, 66, 313, 9, 105, 7, 12274, 7, 13802, 2, 2094, 306, 306, 306, 7, 12274, 7, 238, 574, 66, 13803, 306, 306, 306, 7, 270, 76, 40, 3945, 53, 40, 13804, 7, 13805, 4, 102, 13806, 7, 8, 12413, 7, 270, 76, 13807, 101, 227, 1008, 7, 3116, 3806, 352, 13808, 7, 478, 227, 228, 66, 8658, 306, 306, 306, 7, 1506, 2724, 2, 227, 1825, 13809, 7, 8, 5, 22, 2, 255, 780, 7, 1259, 262, 3055, 96, 55, 36, 19, 306, 306, 306, 7, 5537, 9306, 2, 13795, 1359

---
## PASO 6: Entrenamiento del Modelo

### El ciclo de entrenamiento

El ciclo es **casi idéntico al de la práctica Bengio generativa**, con dos diferencias:

1. **Los logits son 3D** `[batch, seq_len, vocab_size]` y hay que aplanarlos a 2D antes de calcular la pérdida.
2. **`ignore_index=<PAD>`** hace que los tokens de relleno no contribuyan al gradiente.

```
PASO 1 - FORWARD PASS
   La RNN procesa toda la secuencia de una vez.
   Produce un logit de vocab_size para cada posición t.

PASO 2 - CÁLCULO DE LA PÉRDIDA
   CrossEntropyLoss compara cada logit[t] con target[t].
   Los tokens <PAD> se ignoran (ignore_index).

PASO 3 - BACKWARD PASS
   Los gradientes se propagan hacia atrás a través del tiempo
   (BPTT — Backpropagation Through Time).

PASO 4 - ACTUALIZACIÓN DE PESOS
   Adam actualiza Wₓ, Wₕ, b y la tabla de embeddings.
```

### Perplejidad como métrica

La perplejidad mide cuánto de "sorprendido" está el modelo ante el corpus:

```
Perplexity = e^(loss)
```

- Perplejidad del vocabulario completo (azar puro): `e^ln(vocab_size) = vocab_size`
- Perplejidad de 1.0: predicción perfecta (memorización total)
- **Objetivo en práctica**: que baje con las épocas, sin llegar a 1 (sobreajuste)

### Aplanar logits y targets antes de la pérdida

```python
# Logits: [batch, seq_len, vocab_size]  ->  [batch*seq_len, vocab_size]
outputs_flat = outputs.view(-1, vocab.size)

# Targets: [batch, seq_len]             ->  [batch*seq_len]
targets_flat = targets.view(-1)
```

`CrossEntropyLoss` espera exactamente estas formas.


In [11]:
def train_model(model, data_loader, vocab, num_epochs, learning_rate, device='cpu'):
    """
    Entrena el modelo de lenguaje ElmanRNN.

    Args:
        model         : Modelo ElmanRNN.
        data_loader   : DataLoader con los datos de entrenamiento.
        vocab         : Vocabulario (para ignore_index de <PAD>).
        num_epochs    : Número de vueltas completas al dataset.
        learning_rate : Tasa de aprendizaje para Adam.
        device        : 'cpu' o 'cuda'.
    """
    model.to(device)
    model.train()

    # CrossEntropyLoss con ignore_index: los tokens <PAD> no contribuyen al error
    # (Igual que en la práctica Bengio generativa)
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.token_to_idx['<PAD>'])

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f'Iniciando entrenamiento en {device}')
    print(f'  Épocas: {num_epochs}  |  LR: {learning_rate}  |  Batch: {data_loader.batch_size}')
    print('-' * 65)

    for epoch in range(num_epochs):
        total_loss = 0.0

        for inputs, targets in data_loader:
            inputs  = inputs.to(device)    # [batch, seq_len]
            targets = targets.to(device)   # [batch, seq_len]

            optimizer.zero_grad()

            # ── PASO 1: Forward pass ──────────────────────────────────────
            # hidden=None -> PyTorch inicializa el estado oculto a ceros
            outputs, _ = model(inputs)    # [batch, seq_len, vocab_size]

            # ── PASO 2: Aplanar y calcular pérdida ────────────────────────
            # CrossEntropyLoss espera [N, C] y [N]
            outputs_flat = outputs.view(-1, vocab.size)   # [batch*seq_len, vocab_size]
            targets_flat = targets.view(-1)               # [batch*seq_len]
            loss = criterion(outputs_flat, targets_flat)

            # ── PASO 3: Backward pass ─────────────────────────────────────
            loss.backward()    # BPTT — gradientes a través del tiempo

            # ── PASO 4: Actualización de pesos ────────────────────────────
            optimizer.step()

            total_loss += loss.item()

        avg_loss   = total_loss / len(data_loader)
        perplexity = torch.exp(torch.tensor(avg_loss)).item()
        print(f'Época [{epoch+1:3d}/{num_epochs}]  |  Loss: {avg_loss:.4f}  |  Perplejidad: {perplexity:.2f}')

    print('-' * 65)
    print('Entrenamiento completado.')


NUM_EPOCHS    = 10
LEARNING_RATE = 0.001
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_model(
    model         = model,
    data_loader   = rnn_loader,
    vocab         = vocab,
    num_epochs    = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    device        = device
)


Iniciando entrenamiento en cuda
  Épocas: 10  |  LR: 0.001  |  Batch: 4
-----------------------------------------------------------------
Época [  1/10]  |  Loss: 6.2884  |  Perplejidad: 538.31
Época [  2/10]  |  Loss: 5.3108  |  Perplejidad: 202.51
Época [  3/10]  |  Loss: 4.8861  |  Perplejidad: 132.44
Época [  4/10]  |  Loss: 4.5570  |  Perplejidad: 95.30
Época [  5/10]  |  Loss: 4.2522  |  Perplejidad: 70.26
Época [  6/10]  |  Loss: 3.9849  |  Perplejidad: 53.78
Época [  7/10]  |  Loss: 3.7448  |  Perplejidad: 42.30
Época [  8/10]  |  Loss: 3.5188  |  Perplejidad: 33.74
Época [  9/10]  |  Loss: 3.3124  |  Perplejidad: 27.45
Época [ 10/10]  |  Loss: 3.1132  |  Perplejidad: 22.49
-----------------------------------------------------------------
Entrenamiento completado.


---
## PASO 7: Generación de Texto con Muestreo Top-K Autoregresivo

### ¿Cómo genera texto la RNN?

La generación es **autoregresiva**: el modelo predice el siguiente token, lo añade a la secuencia, y vuelve a predecir.

```
Semilla: ['Ya']

Paso 1:  input='Ya'      hidden=None -> predice 'te'    hidden=h₁
Paso 2:  input='te'      hidden=h₁   -> predice 'dije'  hidden=h₂
Paso 3:  input='dije'    hidden=h₂   -> predice 'frases' hidden=h₃
...
Resultado: 'Ya te dije frases hechas y deshechas <EOL> ...'
```

### Ventaja sobre Bengio

La RNN **reutiliza el estado oculto entre pasos** en lugar de procesar toda la historia desde cero:
- Bengio: en cada paso de generación recodifica los últimos N tokens.
- **RNN**: en cada paso solo procesa el último token y actualiza `hidden`.

### Muestreo Top-K (igual que en la práctica anterior)

1. Seleccionamos los **K tokens con mayor logit**.
2. Normalizamos con Softmax → probabilidades.
3. Muestreamos aleatoriamente según esas probabilidades.

| K | Comportamiento |
|---|---|
| 1 | Greedy: determinista, puede ser repetitivo |
| 3–10 | Balance entre coherencia y variedad |
| 50+ | Muy creativo pero puede ser incoherente |


In [12]:
def generate_language_with_top_k_sampling(model, start_text, max_length, k, vocab, device='cpu'):
    """
    Genera texto usando el modelo ElmanRNN con muestreo Top-K.

    El proceso es autoregresivo y STATEFUL: el estado oculto se propaga
    entre pasos de generación, acumulando la historia completa.

    Reutilizada de ElmanRNN.py con adaptaciones para el formato de corpus.

    Args:
        model      : Modelo ElmanRNN ya entrenado.
        start_text : Lista de tokens semilla. Ej: ['Ya'].
        max_length : Número máximo de tokens adicionales a generar.
        k          : Candidatos para el muestreo Top-K.
        vocab      : Vocabulario.
        device     : 'cpu' o 'cuda'.

    Returns:
        str: Texto generado (tokens separados por espacios).
    """
    # Codificamos la semilla
    input_tokens = vocab.encode(start_text)
    pad_idx      = vocab.token_to_idx['<PAD>']

    model.to(device)
    model.eval()

    with torch.no_grad():
        hidden = None   # El estado oculto se inicializa a ceros

        for _ in range(max_length):
            # Tensor de entrada: [1, len(input_tokens)]
            inputs = torch.tensor([input_tokens], device=device)

            # Forward pass — obtenemos logits Y actualizamos el estado oculto
            outputs, hidden = model(inputs, hidden)

            # Logit del ÚLTIMO token de la secuencia actual
            logits = outputs[0, -1, :]   # [vocab_size]

            # ── Muestreo Top-K ─────────────────────────────────────────────
            top_k_logits, top_k_indices = torch.topk(logits, k)
            probs          = F.softmax(top_k_logits, dim=-1)
            sampled_pos    = torch.multinomial(probs, 1).item()
            next_token     = top_k_indices[sampled_pos].item()

            input_tokens.append(next_token)

            # Paramos si generamos <PAD> (fin implícito de secuencia)
            if next_token == pad_idx:
                break

    return ' '.join(vocab.decode(input_tokens))


print('Texto generado por el modelo Elman RNN:')
print('=' * 60)

semillas = ['<PAD>'], ['<EOL>'], ['amigo'], ['amiga']

for semilla in semillas:
    palabra = semilla[0]
    if palabra in vocab.token_to_idx:
        texto = generate_language_with_top_k_sampling(
            model      = model,
            start_text = semilla,
            max_length = 25,
            k          = 10,
            vocab      = vocab,
            device     = device
        )
        print(f'\n[{palabra}]')
        print(f'  {texto}')
    else:
        print(f'\n[{palabra}] -> Palabra no está en el vocabulario')


Texto generado por el modelo Elman RNN:

[<PAD>]
  <PAD> mañana no se puede <EOL> Porque a mi lado , ni lo sabe <EOL> Te digo que no se rompen <EOL> Que no se deja

[<EOL>]
  <EOL> Para llevarte a la vida <EOL> Pero no es mi alma , mi amor <EOL> Intentaré más , palabras más <EOL> No se puede vivir

[amigo]
  amigo desconocido el día contando en la libertad <EOL> Y no madura más tendremos <EOL> No tengo que no se detiene <EOL> Y te espero en

[amiga]
  amiga sólo <EOL> Sé que no te quiero <EOL> Ya no me importa <EOL> Y si pregunto es que no es lo mismo <EOL> Pero no


---
## Exploración: Visualización del Estado Oculto

Una ventaja única de la RNN sobre Bengio es el **estado oculto explícito**.  
Podemos inspeccionar cómo evoluciona `hₜ` a medida que procesamos una frase.


In [13]:
# Procesamos una canción y guardamos el estado oculto en cada paso de tiempo
cancion_ejemplo = sentences[0]
encoded_ejemplo = vocab.encode(cancion_ejemplo)

model.eval()
hidden_states = []

with torch.no_grad():
    hidden = None
    for t, token_idx in enumerate(encoded_ejemplo[:-1]):
        x_t = torch.tensor([[token_idx]], device=device)   # [1, 1]
        _, hidden = model(x_t, hidden)
        # Guardamos la norma L2 del estado oculto como proxy de su "activación"
        hidden_norm = hidden.squeeze().norm().item()
        hidden_states.append((cancion_ejemplo[t], hidden_norm))

print('Evolución del estado oculto (norma L2 de hₜ):')
print(f'  {"TOKEN":20s}  NORMA h')
print('  ' + '-' * 35)
for token, norm in hidden_states[:15]:
    bar = '█' * int(norm / max(n for _, n in hidden_states) * 20)
    print(f'  {token:20s}  {norm:6.2f}  {bar}')
if len(hidden_states) > 15:
    print(f'  ... ({len(hidden_states) - 15} tokens más)')


Evolución del estado oculto (norma L2 de hₜ):
  TOKEN                 NORMA h
  -----------------------------------
  Tú                     11.80  ████████████████
  y                      13.48  ███████████████████
  yo                     13.84  ███████████████████
  en                     13.88  ███████████████████
  mi                     13.98  ███████████████████
  habitación             13.55  ███████████████████
  <EOL>                  13.46  ███████████████████
  Y                      13.76  ███████████████████
  el                     13.72  ███████████████████
  mundo                  14.06  ███████████████████
  fuera                  12.90  ██████████████████
  <EOL>                  13.70  ███████████████████
  Puede                  13.31  ██████████████████
  que                    13.60  ███████████████████
  parezca                13.61  ███████████████████
  ... (168 tokens más)


---
## Cuestionario de Reflexión

---

**Pregunta 1:** La RNN de Elman usa `tanh` como activación, mientras que Bengio usa `ReLU`. ¿Qué ventaja tiene `tanh` para el estado oculto recurrente? ¿Qué problema introduce en secuencias muy largas?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 2:** En el entrenamiento, inicializamos `hidden=None` al principio de cada batch (inicio del forward pass), por lo que PyTorch pone el estado a cero. ¿Sería mejor pasar el estado oculto **entre batches** (entrenamiento stateful)? ¿Qué ventajas e inconvenientes tendría?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 3:** Compara la estrategia de dataset: Bengio genera `Σ len(canción)` ejemplos (uno por token), mientras que la RNN genera `len(canciones)` ejemplos (uno por canción). ¿Cuál entrena con más diversidad de contextos por época? ¿Cuál aprovecha mejor la GPU?

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 4:** El padding con `<PAD>` se ignora en la pérdida gracias a `ignore_index`. Sin embargo, el estado oculto **sí** se actualiza al procesar tokens `<PAD>`. ¿Qué consecuencia tiene esto? ¿Cómo se podría evitar? *(Pista: investiga `PackedSequence` en PyTorch.)*

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 5:** La RNN tiene un parámetro adicional sobre Bengio: la matriz `Wₕ` (`hidden_dim × hidden_dim`). Para `hidden_dim=256`, ¿cuántos parámetros adicionales supone? Calcula también los parámetros totales y compáralos con el modelo Bengio de la práctica anterior.

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---

**Pregunta 6 (Investigación):** La RNN de Elman sufre el problema del **gradiente evanescente** en secuencias largas. ¿Qué arquitecturas se propusieron para resolverlo? Describe brevemente cómo lo consiguen.

**Tu respuesta:**
*(Escribe tu respuesta aquí...)*

---


---
## Extensión opcional: Experimenta con los hiperparámetros

Modifica los valores y vuelve a ejecutar el notebook completo desde el Paso 4.

| Hiperparámetro | Valor base | Efecto al aumentar |
|---|---|---|
| `EMBED_SIZE` | 128 | Representaciones más ricas |
| `HIDDEN_DIM` | 256 | Mayor memoria y capacidad |
| `NUM_EPOCHS` | 30 | Más entrenamiento (riesgo de sobreajuste) |
| `LEARNING_RATE` | 0.01 | Convergencia más rápida (riesgo de inestabilidad) |
| `BATCH_SIZE` | 4 | Mayor batch = gradientes más estables |

### Variante avanzada: RNN multicapa

PyTorch permite apilar varias RNN con el parámetro `num_layers`:

```python
self.rnn = nn.RNN(embed_size, hidden_dim, batch_first=True, num_layers=2)
```

Con `num_layers=2`, la segunda RNN recibe como entrada los estados ocultos de la primera.  
¿Mejora la perplejidad final? ¿A qué coste?


In [14]:
# ── ZONA DE EXPERIMENTOS ──────────────────────────────────────────────────────
EMBED_SIZE    = 128
HIDDEN_DIM    = 256
NUM_EPOCHS    = 30
LEARNING_RATE = 0.01
BATCH_SIZE    = 4

print('Valores actuales:')
print(f'  EMBED_SIZE    = {EMBED_SIZE}')
print(f'  HIDDEN_DIM    = {HIDDEN_DIM}')
print(f'  NUM_EPOCHS    = {NUM_EPOCHS}')
print(f'  LEARNING_RATE = {LEARNING_RATE}')
print(f'  BATCH_SIZE    = {BATCH_SIZE}')
print('\nVuelve a ejecutar desde el Paso 4 para aplicar los cambios.')


Valores actuales:
  EMBED_SIZE    = 128
  HIDDEN_DIM    = 256
  NUM_EPOCHS    = 30
  LEARNING_RATE = 0.01
  BATCH_SIZE    = 4

Vuelve a ejecutar desde el Paso 4 para aplicar los cambios.
